In [1]:
#use "../lib/nb.ml";;
load_css ()

- : unit = ()
Findlib has been successfully loaded. Additional directives:
  #require "package";;      to load a package
  #list;;                   to list the available packages
  #camlp4o;;                to load camlp4 (standard syntax)
  #camlp4r;;                to load camlp4 (revised syntax)
  #predicates "p,q,...";;   to set these predicates
  Topfind.reset();;         to force that packages will be reloaded
  #thread;;                 to enable threads

- : unit = ()
- : unit = ()
val lib_dir : string = "../lib"
val loaded_libs : (string, unit) Hashtbl.t = <abstr>
val load_lib : string -> unit = <fun>


- : unit = ()


# A Simple Compiler for a Fragment of  `C`

This file shows how a simple compiler for a fragment of the programming language `C` can be implemented using the
scanner generator `ocamllex` and the parser generator `Menhir`.  (The Python version of this notebook uses `Lark`.)

First, we describe the grammar of our language using EBNF-notation, since this enables us to give a very concise presentation of the grammar.
Below, the token 'NUMBER' stands for natural numbers and 'ID' represents identifiers.
```
program: function+
       
function: 'int' ID '(' ('int' ID (',' 'int' ID)*)?  ')' '{' decl* stmnt* '}' 
    
decl: 'int' ID ';'  
    
stmnt: '{' stmnt* '}'
     | ID '=' expr ';' 
     | 'if' '(' boolExpr ')' stmnt 
     | 'if' '(' boolExpr ')' stmnt 'else' stmnt 
     | 'while' '(' boolExpr ')' stmnt 
     | 'return' expr ';'
     | expr ';'             
     
boolExpr: boolExpr ('&&' | '||') boolExpr
        | '!' boolExpr             
        | '(' boolExpr ')'
        | expr ('==' | '!=' | '<=' | '>=' | '<' | '>')  expr 
     
expr: expr ('+' | '-' | '*' | '/' | '%') expr 
    | '(' expr ')' 
    | NUMBER
    | ID ('(' (expr (',' expr)*)? ')')?  
```
This grammar is ambiguous.  `Menhir` supports precedence declarations and the notebook
[Compiler-For.ipynb](Compiler-For.ipynb) uses them.  The Python version of this notebook uses `Lark`, which offers no
precedence declarations for its <span style="font-variant: small-caps;">Lalr</span> parser.  Therefore, we will
instead *stratify* the grammar, like the Python version does:  we introduce one syntactical variable per precedence
level.  This removes the ambiguity and, as a side effect, makes the precedences and associativities visible in the
grammar itself.

## Specification of the Scanner

`Lark` reads a *single* grammar that specifies both the *tokens* and the *grammar rules*.  With `ocamllex` and `Menhir`,
the scanner and the parser are specified separately:
- The *scanner specification* for `ocamllex` consists of a *header* in braces, which contains OCaml code, and the rule
  `token`.  Every case of this rule consists of a regular expression and an *action*, which returns the token that has
  been recognized.  The names of the tokens are written in `UPPER_CASE`.
- The *grammar* for `Menhir` declares the tokens and contains the grammar rules.  The names of *syntactical
  variables* are written in `lower_case`.

We build the scanner specification as a string.  In this section we collect the parts of the scanner specification
in the variable `terminal_definitions`, while the next section collects the grammar rules in the variable `syntax`.

Every token carries a value of type `Parse_tree.tok`, which stores the type of the token, the string that has been
recognized, and its position.  The function `Parse_tree.token` creates this value.  We use the names that `Lark` uses
for these tokens.

In [ ]:
load_lib "parser_generator.ml";;
open Parse_tree

The token `NUMBER` specifies a natural number.  In `ocamllex`, characters are enclosed in single quotes and
character sets in brackets.  The header opens the module `C_parser`, which is generated by `Menhir` and defines the
tokens.

In [ ]:
let terminal_definitions = {|
{
  open C_parser
}

rule token = parse
  | '0' | ['1'-'9'] ['0'-'9']*    { NUMBER (Parse_tree.token "NUMBER" lexbuf) }
|}

Next, we define the tokens for the operator symbols.  In `Lark`, a string that occurs literally inside a grammar rule,
for example `"=="`, is turned into an *anonymous terminal* automatically and only the operators consisting of more than
one character need a declaration in order to get a readable name.  `ocamllex` and `Menhir` have no anonymous
terminals, so every operator has to be defined.  We use the names `EQ`, `NE`, `LE`, `GE`, `AND`, and `OR` of the
`Lark` version for the operators consisting of two characters and the names that `Lark` invents for the anonymous
terminals, i.e. `PLUS`, `MINUS`, `STAR`, `LPAR`, `RPAR`, and so on, for the other operators.

Note that a string can be used as a regular expression.  In this case no escaping is needed:  we can simply write
`"||"`.  Furthermore, `ocamllex` always recognizes the *longest* string that matches, so `"!="` is preferred over `"!"`
and `"<="` is preferred over `"<"`.

In [ ]:
let terminal_definitions = terminal_definitions ^ {|
  | "=="                          { EQ        (Parse_tree.token "EQ"        lexbuf) }
  | "!="                          { NE        (Parse_tree.token "NE"        lexbuf) }
  | "<="                          { LE        (Parse_tree.token "LE"        lexbuf) }
  | ">="                          { GE        (Parse_tree.token "GE"        lexbuf) }
  | "&&"                          { AND       (Parse_tree.token "AND"       lexbuf) }
  | "||"                          { OR        (Parse_tree.token "OR"        lexbuf) }
  | '+'                           { PLUS      (Parse_tree.token "PLUS"      lexbuf) }
  | '-'                           { MINUS     (Parse_tree.token "MINUS"     lexbuf) }
  | '*'                           { STAR      (Parse_tree.token "STAR"      lexbuf) }
  | '/'                           { SLASH     (Parse_tree.token "SLASH"     lexbuf) }
  | '%'                           { PERCENT   (Parse_tree.token "PERCENT"   lexbuf) }
  | '('                           { LPAR      (Parse_tree.token "LPAR"      lexbuf) }
  | ')'                           { RPAR      (Parse_tree.token "RPAR"      lexbuf) }
  | '{'                           { LBRACE    (Parse_tree.token "LBRACE"    lexbuf) }
  | '}'                           { RBRACE    (Parse_tree.token "RBRACE"    lexbuf) }
  | ';'                           { SEMICOLON (Parse_tree.token "SEMICOLON" lexbuf) }
  | ','                           { COMMA     (Parse_tree.token "COMMA"     lexbuf) }
  | '='                           { EQUAL     (Parse_tree.token "EQUAL"     lexbuf) }
  | '<'                           { LESSTHAN  (Parse_tree.token "LESSTHAN"  lexbuf) }
  | '>'                           { MORETHAN  (Parse_tree.token "MORETHAN"  lexbuf) }
  | '!'                           { BANG      (Parse_tree.token "BANG"      lexbuf) }
|}

Our version of `C` allows both *single-line comments* and *multi-line comments*.
- The regular expression `"/*" ([^ '*'] | '*'+ [^ '*' '/'])* '*'+ '/'` recognizes multi-line comments.
  Multi-line comments start with the string `/*` and end with the string `*/`.  The Python version uses the regular
  expression `/\*(.|\n)*?\*/` with the *non-greedy* quantor `*?`.  If we have code like
  ```
  /* blah */ a = 1; /* blub */
  ```
  the greedy quantor would recognize the whole line as one comment.  `ocamllex` has no non-greedy quantors.
  Therefore, our regular expression states explicitly that the comment contains no `*/` before its end.
- The regular expression `"//" [^ '\n']*` recognizes single-line comments.
  A single line comment starts with the string `//` and extends to the end of the line.

The action `token lexbuf` calls the scanner recursively and hence discards the comment, i.e. comments never show up in
the token stream.  This is the `ocamllex` equivalent of the directive `%ignore` of `Lark`.  As a multi-line comment
contains newlines, the function `Parse_tree.new_lines` updates the line number of the scanner.

In [ ]:
let terminal_definitions = terminal_definitions ^ {|
  | "/*" ([^ '*'] | '*'+ [^ '*' '/'])* '*'+ '/'
                                  { Parse_tree.new_lines lexbuf; token lexbuf }
  | "//" [^ '\n']*                { token lexbuf }
|}

The keywords `'int'`, `'if'`, `'else'`, `'while'`, and `'return'` are syntactically identical to identifiers.
In the `Ply` version of this notebook this problem had been solved via the dictionary
```
Keywords = { 'int'   : 'INT', 
             'if'    : 'IF',
             'else'  : 'ELSE', 
             'while' : 'WHILE', 
             'return': 'RETURN'
           }
```
and the function `t_ID` that looked up every identifier in this dictionary.

`Lark` performs this test automatically via the so called *unless* mechanism:  If a string terminal like `"if"` is also
matched by the regular expression of another terminal, in our case by the regular expression defining `NAME`, then
`Lark` first scans a `NAME` and afterwards checks whether the resulting string is one of the keywords.

`ocamllex` has no such mechanism, but the same effect is easily achieved:  the action of the rule for `NAME` checks
whether the string that has been scanned is one of the keywords and, if so, returns the corresponding token.  Because
of this, the string `iffy` is scanned as a single `NAME` and not as the keyword `if` followed by the identifier `fy`.
(If we had used a separate rule for every keyword, the string `if` would be matched by two rules with the same
length.  In this case, `ocamllex` would prefer the rule that is given first.)

In [ ]:
let terminal_definitions = terminal_definitions ^ {|
  | ['a'-'z' 'A'-'Z'] ['a'-'z' 'A'-'Z' '0'-'9' '_']*
      { match Lexing.lexeme lexbuf with
        | "int"    -> INT    (Parse_tree.token "INT"    lexbuf)
        | "if"     -> IF     (Parse_tree.token "IF"     lexbuf)
        | "else"   -> ELSE   (Parse_tree.token "ELSE"   lexbuf)
        | "while"  -> WHILE  (Parse_tree.token "WHILE"  lexbuf)
        | "return" -> RETURN (Parse_tree.token "RETURN" lexbuf)
        | _        -> NAME   (Parse_tree.token "NAME"   lexbuf) }
|}

White space, i.e. *space characters*, *tabulators*, *carriage returns*, and *newlines* are ignored.
In `Ply` we had needed two declarations for this:  the string `t_ignore` for the first three characters and the
function `t_newline`, which additionally had to increment the line number of the scanner.  `ocamllex` needs the same
two rules: a newline calls `Lexing.new_line`, so that the scanner keeps track of the line numbers.  Since the scanner
also knows the position of the beginning of the current line, it computes the column numbers on its own: every token
`t` provides the fields `t.line` and `t.column`.  Hence there is no need for a function like `find_column`.

Finally, the token `EOF` marks the end of the input and a character that is not covered by any of the regular
expressions raises the exception `Unexpected_input`.  Since `ocamllex` tries the rules in the order given, the rule
`_` has to be the last one.

In [ ]:
let terminal_definitions = terminal_definitions ^ {|
  | [' ' '\t' '\r']+              { token lexbuf }
  | '\n'                          { Lexing.new_line lexbuf; token lexbuf }
  | eof                           { EOF }
  | _                             { raise (Parse_tree.unexpected lexbuf ("No terminal matches " ^ repr (Lexing.lexeme lexbuf))) }
|}

This is the complete specification of our scanner.

In [ ]:
print_string terminal_definitions;;
flush stdout

## Specification of the Parser

`Lark` supports <span style="font-variant: small-caps;">Ebnf</span> directly, so the Python version can use the
quantifiers `*`, `+`, and `?`.  `Menhir` does not support these quantifiers in the same way.  Therefore, the grammar has
to be rewritten:  every list has to be expressed by recursive rules like `ne_parameters`, which return OCaml lists.
Furthermore, there are two things that we have to take care of:
1. The rules for `expr` and `bool_expr` are ambiguous.  As in the Python version, we encode the operator precedences
   into the grammar by introducing one syntactical variable per precedence level.
2. Every rule is followed by a *semantic action*, which builds the parse tree.  The parse tree has the type
   `Parse_tree.tree` and resembles the parse tree of `Lark`:  the action `{ Tree ("block", [s]) }` creates a node
   with the name `block` and the child `s`.  These names correspond to the names that are attached to the alternatives
   in the `Lark` grammar with the operator `->`.

The grammar starts with a header that opens the module `Parse_tree` and declares the tokens.

A program is a non-empty list of function definitions.  `Menhir` needs a start rule that is followed by the end of
the input `EOF`.  In the rule `function_`, the tokens `INT`, `LPAR`, `RPAR`, `LBRACE`, and `RBRACE` do not show up in
the parse tree, since the action does not mention them.  Therefore, the parse tree of a `function_` has exactly four
children:  the name of the function, its parameters, its declarations, and its statements.  (The name `function` is a
keyword of OCaml and can not be used as the name of a syntactical variable.)

In [ ]:
let syntax = {|
%{
  open Parse_tree
%}

%token <Parse_tree.tok> NUMBER NAME EQ NE LE GE AND OR INT IF ELSE WHILE RETURN
%token <Parse_tree.tok> PLUS MINUS STAR SLASH PERCENT LPAR RPAR LBRACE RBRACE SEMICOLON COMMA EQUAL LESSTHAN MORETHAN BANG
%token EOF

%start <Parse_tree.tree> program
%type  <Parse_tree.tree> function_ parameters declarations statements statement
%type  <Parse_tree.tree> bool_expr conjunction negation bool_atom expr product factor arguments
%type  <Parse_tree.tree list> functions ne_parameters declaration_list statement_list ne_arguments

%%

program: fs = functions EOF                       { Tree ("program", fs) }

functions: f = function_                          { [f] }
         | f = function_ fs = functions           { f :: fs }

function_: INT n = NAME LPAR ps = parameters RPAR LBRACE ds = declarations ss = statements RBRACE
                                                  { Tree ("function", [Token n; ps; ds; ss]) }
|}

The syntactical variables `parameters`, `declarations`, and `statements` describe lists.  As `Menhir` does not support
<span style="font-variant: small-caps;">Ebnf</span>, these lists are defined via recursion.  Note that the parse trees
of `parameters` and `declarations` contain only the tokens of type `NAME`, since the actions do not mention the other
tokens.

In [ ]:
let syntax = syntax ^ {|
parameters  : /* empty */                         { Tree ("parameters", []) }
            | ps = ne_parameters                  { Tree ("parameters", ps) }

ne_parameters: INT n = NAME                       { [Token n] }
             | INT n = NAME COMMA ps = ne_parameters
                                                  { Token n :: ps }

declarations: ds = declaration_list               { Tree ("declarations", ds) }

declaration_list: /* empty */                     { [] }
                | INT n = NAME SEMICOLON ds = declaration_list
                                                  { Token n :: ds }

statements  : ss = statement_list                 { Tree ("statements", ss) }

statement_list: /* empty */                       { [] }
              | s = statement ss = statement_list { s :: ss }
|}

The rule `statement` has seven alternatives.  The names of the nodes that are created by the actions will later be
used to build the abstract syntax tree for the corresponding alternative.

The two alternatives `if_stmnt` and `if_else_stmnt` cause the well-known 
[dangling else problem](https://en.wikipedia.org/wiki/Dangling_else):  when the parser has read a statement of the
form `if ( bool_expr ) statement` and the next token is the keyword `else`, then the parser can either *reduce* this
statement or it can *shift* the token `else`.  In the `Ply` version of this notebook this *shift/reduce conflict*
had been resolved by declaring the precedence of the token `ELSE` to be higher than the precedence of the token
`IF` and by attaching the precedence of `IF` to the rule `stmnt : IF '(' bool_expr ')' stmnt` via the keyword
`%prec`.  `Menhir` supports the same declarations, see [Compiler-For.ipynb](Compiler-For.ipynb).

If no precedence is declared, `Menhir` resolves a shift/reduce conflict *in favour of shifting*, like `Lark` does.
This is exactly what we want:  the keyword `else` is attached to the innermost `if` that does not yet have an `else`
part.  Hence a program of the form
```
if (a == 1) if (b == 1) x = 2; else x = 3;
```
is interpreted as
```
if (a == 1) {
    if (b == 1) {
        x = 2;
    } else {
        x = 3;
    }
}
```
and not as
```
if (a == 1) {
    if (b == 1) {
        x = 2;
    } 
} else {
    x = 3;
}
```

In [ ]:
let syntax = syntax ^ {|
statement: LBRACE ss = statements RBRACE          { Tree ("block", [ss]) }
         | n = NAME EQUAL e = expr SEMICOLON      { Tree ("assign", [Token n; e]) }
         | IF LPAR b = bool_expr RPAR s = statement
                                                  { Tree ("if_stmnt", [b; s]) }
         | IF LPAR b = bool_expr RPAR s1 = statement ELSE s2 = statement
                                                  { Tree ("if_else_stmnt", [b; s1; s2]) }
         | WHILE LPAR b = bool_expr RPAR s = statement
                                                  { Tree ("while_stmnt", [b; s]) }
         | RETURN e = expr SEMICOLON              { Tree ("return_stmnt", [e]) }
         | e = expr SEMICOLON                     { Tree ("expr_stmnt", [e]) }
|}

The `Ply` version of this notebook had used the precedence declarations
```
('left'    , 'OR'),
('left'    , 'AND'),
('right'   , '!'),
('nonassoc', 'EQ', 'NE', 'LE', 'GE', '<', '>'),
```
to disambiguate the rules for `bool_expr`.  We express the same information by *stratifying* the grammar:
- `bool_expr` is a list of `conjunction`s that are separated by the operator `||`,
- a `conjunction` is a list of `negation`s that are separated by the operator `&&`,
- a `negation` is a `bool_atom` that is preceded by an arbitrary number of `!` operators,
- a `bool_atom` is either a comparison of two arithmetical expressions or a `bool_expr` in parentheses.

Since the operators `||` and `&&` are *left associative*, the corresponding rules are *left recursive*:  the
syntactical variable on the left hand side of the rule occurs as the *first* symbol on the right hand side.  The
operator `!` is *right associative*, hence the rule for `not_expr` is *right recursive*.  As the comparison operators
are non-associative, the rule `bool_atom` is not recursive at all:  an expression like `a < b < c` is a syntax
error.

The `Lark` grammar uses a question mark in front of the names `bool_expr`, `conjunction`, `negation`, and `bool_atom` in
order to *inline* these syntactical variables.  In `Menhir`, the action of an alternative consisting of a single
syntactical variable simply returns the parse tree of this variable, e.g. `{ c }`.  Without this, the parse tree of the
expression `x == 1` would contain a chain of four nodes.

In [ ]:
let syntax = syntax ^ {|
bool_expr: b = bool_expr OR c = conjunction       { Tree ("or_expr", [b; c]) }
         | c = conjunction                        { c }

conjunction: c = conjunction AND n = negation     { Tree ("and_expr", [c; n]) }
           | n = negation                         { n }

negation: BANG n = negation                       { Tree ("not_expr", [n]) }
        | a = bool_atom                           { a }

bool_atom: l = expr EQ       r = expr             { Tree ("equal",            [l; r]) }
         | l = expr NE       r = expr             { Tree ("unequal",          [l; r]) }
         | l = expr LE       r = expr             { Tree ("less_or_equal",    [l; r]) }
         | l = expr GE       r = expr             { Tree ("greater_or_equal", [l; r]) }
         | l = expr LESSTHAN r = expr             { Tree ("less",             [l; r]) }
         | l = expr MORETHAN r = expr             { Tree ("greater",          [l; r]) }
         | LPAR b = bool_expr RPAR                { b }
|}

The arithmetical expressions are stratified in the same way.  The precedence declarations
```
('left', '+', '-'),
('left', '*', '/', '%')
```
of the `Ply` version are replaced by the three levels `expr`, `product`, and `factor`.  Again, all rules are left
recursive because all arithmetical operators are left associative.

The last alternative of the rule `factor` creates no node of its own:  a parenthesized expression is represented by
the parse tree of the expression inside the parentheses.

In [ ]:
let syntax = syntax ^ {|
expr: e = expr PLUS  p = product                  { Tree ("add",      [e; p]) }
    | e = expr MINUS p = product                  { Tree ("subtract", [e; p]) }
    | p = product                                 { p }

product: p = product STAR    f = factor           { Tree ("multiply", [p; f]) }
       | p = product SLASH   f = factor           { Tree ("divide",   [p; f]) }
       | p = product PERCENT f = factor           { Tree ("modulo",   [p; f]) }
       | f = factor                               { f }

factor: n = NUMBER                                { Tree ("number",        [Token n]) }
      | n = NAME                                  { Tree ("variable",      [Token n]) }
      | n = NAME LPAR a = arguments RPAR          { Tree ("function_call", [Token n; a]) }
      | LPAR e = expr RPAR                        { e }

arguments: /* empty */                            { Tree ("arguments", []) }
         | es = ne_arguments                      { Tree ("arguments", es) }

ne_arguments: e = expr                            { [e] }
            | e = expr COMMA es = ne_arguments    { e :: es }
|}

The complete grammar consists of the grammar rules.  The terminal definitions are given separately to `ocamllex`.

In [ ]:
let grammar = syntax;;
print_string grammar;;
flush stdout

Now we can create the scanner and the parser.  The function `generate_parser` runs `menhir --lalr`, which creates an
<span style="font-variant: small-caps;">Lalr</span> parser like the keyword argument `parser='lalr'` of `Lark`.  The
start variable is declared in the grammar via `%start`.

`Menhir` always reports shift/reduce conflicts and reduce/reduce conflicts.  Because of the option `--explain`, the
conflicts are explained in the file `_build/c_parser.conflicts`.  The states of the automaton are written to the file
`_build/c_parser.automaton`.  This is the `Menhir` equivalent of the file `parser.out` that had been produced by `Ply`.

In [ ]:
(* Menhir reports its conflicts without further ado. *)

In [ ]:
generate_parser ~name:"c" ~lexer:terminal_definitions ~grammar

The only conflict that is reported is the shift/reduce conflict that results from the *dangling else
ambiguity*.  As discussed above, `Menhir` resolves this conflict in favour of shifting and hence in the way that we
want.  All other conflicts have been eliminated by stratifying the grammar.

In [ ]:
cat "_build/c_parser.conflicts"

## Testing the Scanner

The function `lex` runs the scanner without invoking the parser.  It returns the list of tokens.
Every token `t` knows its type `t.kind`, its value `t.value`, and its position `t.line` and `t.column`.  Note that
white space and comments do not show up in the resulting list of tokens because the scanner discards them, and that
the keywords have the types `INT`, `IF`, `ELSE`, `WHILE`, and `RETURN` instead of `NAME`.

**Type safety:** `Lark` represents the type of a token as a string.  Here, the tokens are values of the variant type
`C_parser.token`, which is generated by `Menhir`.  The compiler checks that `lex` handles every kind of token.

In [ ]:
let lex (program : string) : tok list =
  let lexbuf = Lexing.from_string program in
  let rec loop () =
    match C_lexer.token lexbuf with
    | C_parser.EOF -> []
    | NUMBER t | NAME t | EQ t | NE t | LE t | GE t | AND t | OR t | INT t | IF t | ELSE t | WHILE t | RETURN t
    | PLUS t | MINUS t | STAR t | SLASH t | PERCENT t | LPAR t | RPAR t | LBRACE t | RBRACE t | SEMICOLON t
    | COMMA t | EQUAL t | LESSTHAN t | MORETHAN t | BANG t ->
        t :: loop ()
  in
  loop ()

let test_scanner (file_name : string) : tok list =
  let program = read_file file_name in
  print_endline program;
  lex program

In [ ]:
List.iter (fun t -> Printf.printf "%3d, %3d: %-10s %s\n%!" t.line t.column t.kind t.value) (test_scanner "Examples/MySum.c")

## Construction of the Abstract Syntax Tree

The parse tree that is computed by the parser has the type `Parse_tree.tree`.  As the compiler that is developed below
works on *abstract syntax trees*, we have to convert the parse tree into an abstract syntax tree.  In `Ply`, this
conversion had been done by the functions `p_program_one`, `p_function`, and so on:  every grammar rule had its own
function that computed the abstract syntax tree of the left hand side from the abstract syntax trees of the symbols
on the right hand side.

In `Lark`, the same is achieved with a `Transformer`, which has a method for every name of a node.  In OCaml, the
functions `build_expr`, `build_bool`, `build_stmnt`, `build_function`, and `build_program` play this role:  they
traverse the parse tree and have a case for every name of a node.

The Python version represents abstract syntax trees as *nested tuples*:
- a list of *n* items is represented as `('.', item₁, ⋯, itemₙ)`,
- a function definition is represented as `('fct', name, parameters, variables, statements)`,
- a variable is represented as a string, while a number is represented as `('Number', digits)`,
- all other nodes are tuples whose first component is the operator.

In OCaml, we define a type for every kind of abstract syntax tree instead:  `expr` for arithmetical expressions,
`bool_expr` for Boolean expressions, `stmnt` for statements, `fct` for function definitions, and `program` for
programs.  A list of *n* items is an OCaml list and a function definition is a record.  The function `program_to_nested`,
which is defined below, converts an abstract syntax tree into the nested tuple of the Python version.

**Type safety:** The Python functions that generate code have to check the shape of every nested tuple at runtime.  In
OCaml, the types guarantee that, e.g., the condition of a `while` loop is a Boolean expression and the arguments of a
function call are arithmetical expressions.

In [ ]:
type arith_op = Add | Sub | Mul | Div | Mod
type cmp_op   = Eq | Ne | Le | Ge | Lt | Gt

type expr =
  | Var    of string                      (* a variable *)
  | Number of string                      (* the digits of a number *)
  | Arith  of arith_op * expr * expr
  | Call   of string * expr list          (* a function call *)

type bool_expr =
  | Compare of cmp_op * expr * expr
  | And     of bool_expr * bool_expr
  | Or      of bool_expr * bool_expr
  | Not     of bool_expr

type stmnt =
  | Block      of stmnt list
  | Assign     of string * expr
  | If         of bool_expr * stmnt
  | If_else    of bool_expr * stmnt * stmnt
  | While      of bool_expr * stmnt
  | Return     of expr
  | Expr_stmnt of expr

type fct = { name : string; parameters : string list; variables : string list; stmnts : stmnt list }

type program = fct list

let unexpected (t : tree) = invalid_arg ("unexpected node in the parse tree: " ^ to_string t)

let name (t : tree) : string = match t with Token tok -> tok.value | _ -> unexpected t

let rec build_expr (t : tree) : expr =
  match t with
  (* arithmetical expressions *)
  | Tree ("add",      [ lhs; rhs ]) -> Arith (Add, build_expr lhs, build_expr rhs)
  | Tree ("subtract", [ lhs; rhs ]) -> Arith (Sub, build_expr lhs, build_expr rhs)
  | Tree ("multiply", [ lhs; rhs ]) -> Arith (Mul, build_expr lhs, build_expr rhs)
  | Tree ("divide",   [ lhs; rhs ]) -> Arith (Div, build_expr lhs, build_expr rhs)
  | Tree ("modulo",   [ lhs; rhs ]) -> Arith (Mod, build_expr lhs, build_expr rhs)
  | Tree ("number",   [ n ])        -> Number (name n)
  | Tree ("variable", [ n ])        -> Var (name n)
  | Tree ("function_call", [ n; Tree ("arguments", exprs) ]) -> Call (name n, List.map build_expr exprs)
  | _ -> unexpected t

let rec build_bool (t : tree) : bool_expr =
  match t with
  (* Boolean expressions *)
  | Tree ("or_expr",  [ lhs; rhs ]) -> Or  (build_bool lhs, build_bool rhs)
  | Tree ("and_expr", [ lhs; rhs ]) -> And (build_bool lhs, build_bool rhs)
  | Tree ("not_expr", [ arg ])      -> Not (build_bool arg)
  | Tree ("equal",            [ lhs; rhs ]) -> Compare (Eq, build_expr lhs, build_expr rhs)
  | Tree ("unequal",          [ lhs; rhs ]) -> Compare (Ne, build_expr lhs, build_expr rhs)
  | Tree ("less_or_equal",    [ lhs; rhs ]) -> Compare (Le, build_expr lhs, build_expr rhs)
  | Tree ("greater_or_equal", [ lhs; rhs ]) -> Compare (Ge, build_expr lhs, build_expr rhs)
  | Tree ("less",             [ lhs; rhs ]) -> Compare (Lt, build_expr lhs, build_expr rhs)
  | Tree ("greater",          [ lhs; rhs ]) -> Compare (Gt, build_expr lhs, build_expr rhs)
  | _ -> unexpected t

let rec build_stmnt (t : tree) : stmnt =
  match t with
  (* statements *)
  | Tree ("block", [ Tree ("statements", stmnts) ]) -> Block (List.map build_stmnt stmnts)
  | Tree ("assign", [ n; expr ])                    -> Assign (name n, build_expr expr)
  | Tree ("if_stmnt", [ condition; stmnt ])         -> If (build_bool condition, build_stmnt stmnt)
  | Tree ("if_else_stmnt", [ condition; then_stmnt; else_stmnt ]) ->
      If_else (build_bool condition, build_stmnt then_stmnt, build_stmnt else_stmnt)
  | Tree ("while_stmnt", [ condition; body ])       -> While (build_bool condition, build_stmnt body)
  | Tree ("return_stmnt", [ expr ])                 -> Return (build_expr expr)
  | Tree ("expr_stmnt", [ expr ])                   -> Expr_stmnt (build_expr expr)
  | _ -> unexpected t

(* program and function definitions *)
let build_function (t : tree) : fct =
  match t with
  | Tree ("function", [ n; Tree ("parameters", ps); Tree ("declarations", ds); Tree ("statements", ss) ]) ->
      { name = name n; parameters = List.map name ps; variables = List.map name ds; stmnts = List.map build_stmnt ss }
  | _ -> unexpected t

let build_program (t : tree) : program =
  match t with
  | Tree ("program", functions) -> List.map build_function functions
  | _ -> unexpected t

In [ ]:
let ast_builder : tree -> program = build_program

The function `parse_program` takes a string containing a `C` program, parses it, and returns the abstract
syntax tree of this program as `Some ast`.

If the program contains a syntax error, the parser raises the exception `Unexpected_input`.  In `Ply`, the
corresponding error message had been printed by the function `p_error`.  The exception provides the fields `line`,
`column`, and `position`, while the function `get_context` returns the line of the program that contains the error
together with a marker that points at the offending token.  In this case, `parse_program` returns `None`.

In [ ]:
let parse_program (program : string) : program option =
  match parse_string C_parser.program C_lexer.token program with
  | exception Unexpected_input { line; column; position; _ } ->
      Printf.printf "Syntax error in line %d, column %d:\n%!" line column;
      print_endline (get_context program position);
      None
  | parse_tree -> Some (ast_builder parse_tree)

Below is an example of an erroneous program.

In [ ]:
parse_program "int main() { x = 1 + ; }"

The notebook `AST2Dot.ipynb` provides the function `tuple2dot`.  This function can be used to visualize the
abstract syntax tree that is generated by the function `parse_program`.

In [ ]:
run_notebook "../AST2Dot.ipynb"

The function `program_to_nested` converts an abstract syntax tree into a nested tuple of the type `nested` from the
notebook `AST2Dot.ipynb`.  This is the nested tuple that the Python version computes.  The function `nested_to_string`
from `AST2Dot.ipynb` writes it in the syntax of Python.

In [ ]:
let rec expr_to_nested (e : expr) : nested =
  let arith_op op = match op with Add -> "+" | Sub -> "-" | Mul -> "*" | Div -> "/" | Mod -> "%" in
  match e with
  | Var x -> Str x
  | Number n -> Tuple ("Number", [ Str n ])
  | Arith (op, lhs, rhs) -> Tuple (arith_op op, [ expr_to_nested lhs; expr_to_nested rhs ])
  | Call (f, args) -> Tuple ("call", Str f :: List.map expr_to_nested args)

let rec bool_to_nested (b : bool_expr) : nested =
  let cmp_op op = match op with Eq -> "==" | Ne -> "!=" | Le -> "<=" | Ge -> ">=" | Lt -> "<" | Gt -> ">" in
  match b with
  | Compare (op, lhs, rhs) -> Tuple (cmp_op op, [ expr_to_nested lhs; expr_to_nested rhs ])
  | And (lhs, rhs) -> Tuple ("&&", [ bool_to_nested lhs; bool_to_nested rhs ])
  | Or (lhs, rhs) -> Tuple ("||", [ bool_to_nested lhs; bool_to_nested rhs ])
  | Not arg -> Tuple ("!", [ bool_to_nested arg ])

let rec stmnt_to_nested (s : stmnt) : nested =
  match s with
  | Block stmnts -> Tuple (".", List.map stmnt_to_nested stmnts)
  | Assign (x, e) -> Tuple ("=", [ Str x; expr_to_nested e ])
  | If (b, s) -> Tuple ("if", [ bool_to_nested b; stmnt_to_nested s ])
  | If_else (b, s1, s2) -> Tuple ("if-else", [ bool_to_nested b; stmnt_to_nested s1; stmnt_to_nested s2 ])
  | While (b, s) -> Tuple ("while", [ bool_to_nested b; stmnt_to_nested s ])
  | Return e -> Tuple ("return", [ expr_to_nested e ])
  | Expr_stmnt e -> expr_to_nested e

let fct_to_nested (f : fct) : nested =
  let names l = Tuple (".", List.map (fun x -> Str x) l) in
  Tuple ("fct", [ Str f.name; names f.parameters; names f.variables; Tuple (".", List.map stmnt_to_nested f.stmnts) ])

let program_to_nested (p : program) : nested = Tuple (".", List.map fct_to_nested p)

The function `parse` takes a `file_name` as its sole argument.  The file is read and parsed.
The resulting abstract syntax tree is printed as a nested tuple and visualized using `graphviz`.  In contrast to the
`Ply` version, there is no need to reset a line counter:  every call of `parse_string` creates a new scanner buffer,
hence the line numbers that occur in error messages are always correct.

In [ ]:
let parse (file_name : string) : unit =
  let program = read_file file_name in
  print_endline program;
  match parse_program program with
  | None -> print_endline "None"
  | Some ast ->
      print_endline (nested_to_string (program_to_nested ast));
      Graphviz.display (tuple2dot (program_to_nested ast))

In [ ]:
parse "Examples/MySum.c"

## Code Generation

The function `indent` is used to indent the generated assembler commands by preceding them with 8 space characters. 

In [ ]:
let indent (s : string) : string = String.make 8 ' ' ^ s

The function `compile_expr expr st class_name` takes three arguments:
- `expr` is an *abstract syntax tree* that represents an expression.  
  This abstract syntax tree has the type `expr`.  
- `st` is short for *symbol table*.  This is a dictionary that maps variable
  names to natural numbers.  Given a variable `x`, the number `Dict.find x st` specifies
  the location where the variable `x` is stored on the stack with respect to the 
  local stack frame.
- `class_name` is the name of the class that is to be generated.

The function returns a pair of the form `(cmds, size)`.
- `cmds` is a list of assembler commands,
- `size` is the maximum size of the stack that is needed. 

**Type safety:** The Python function needs a final case `case _: assert False` for nested tuples that are not
expressions.  As the type `expr` contains only expressions, the OCaml compiler checks that every case is handled, and
the final case is not needed.  For the same reason, the Python dictionary `OpToCmd` becomes a `match` on the operator.

In [ ]:
let rec compile_expr (expr : expr) (st : (string, int) Dict.t) (class_name : string) : string list * int =
  match expr with
  | Var var -> (* expr is a variable name *)
      let cmd = indent (Printf.sprintf "iload %d" (Dict.find var st)) in
      ([ cmd ], 1)
  | Number n ->
      let cmd = indent ("ldc " ^ n) in
      ([ cmd ], 1)
  | Arith (op, lhs, rhs) ->
      let l1, sz1 = compile_expr lhs st class_name in
      let l2, sz2 = compile_expr rhs st class_name in
      let op_to_cmd = match op with Add -> "iadd" | Sub -> "isub" | Mul -> "imul" | Div -> "idiv" | Mod -> "irem" in
      let cmd = indent op_to_cmd in
      (l1 @ l2 @ [ cmd ], max sz1 (1 + sz2))
  | Call ("println", args) ->
      let cmd_lst   = ref [ indent "getstatic java/lang/System/out Ljava/io/PrintStream;" ] in
      let stck_size = ref 0 in
      let cnt       = ref 0 in
      List.iter
        (fun arg ->
          let l, sz_arg = compile_expr arg st class_name in
          stck_size := max !stck_size (!cnt + 1 + sz_arg);
          cmd_lst   := !cmd_lst @ l;
          cnt       := !cnt + 1)
        args;
      cmd_lst := !cmd_lst @ [ indent (Printf.sprintf "invokevirtual java/io/PrintStream/println(%s)V" (String.make !cnt 'I')) ];
      (!cmd_lst, !stck_size)
  | Call (f, args) ->
      let cmd_lst   = ref [] in
      let stck_size = ref 0 in
      let cnt       = ref 0 in
      List.iter
        (fun arg ->
          let l, sz_arg = compile_expr arg st class_name in
          stck_size := max !stck_size (!cnt + sz_arg);
          cmd_lst   := !cmd_lst @ l;
          cnt       := !cnt + 1)
        args;
      cmd_lst := !cmd_lst @ [ indent (Printf.sprintf "invokestatic %s/%s(%s)I" class_name f (String.make !cnt 'I')) ];
      (!cmd_lst, max !stck_size 1)

The following is a test of the function `compile_expr`.

In [ ]:
let expr = Call ("sum", [ Arith (Add, Var "x", Var "y") ])
let st   = Dict.of_list [ ("x", 0); ("y", 1) ];;
compile_expr expr st "Sum"

In [ ]:
let expr = Arith (Add, Var "x", Var "y")
let st   = Dict.of_list [ ("x", 0); ("y", 1) ];;
compile_expr expr st "Sum"

The variable `label_counter` is a global counter that is used to create unique label names.
Every call of `new_label` creates a new, unique label.

In [ ]:
let expr = Call ("println", [ Var "x"; Var "y" ])
let st   = Dict.of_list [ ("x", 0); ("y", 1) ];;
compile_expr expr st "Sum"

In [ ]:
let label_counter = ref 0

let new_label () : string =
  label_counter := !label_counter + 1;
  "l" ^ string_of_int !label_counter

The function `compile_bool expr st class_name` takes three arguments:
- `expr` is an *abstract syntax tree* that represents a Boolean expression.  
  This abstract syntax tree has the type `bool_expr`.  
- `st` is short for *symbol table*.  This is a dictionary that maps variable
  names to natural numbers.  Given a variable `x`, the number `Dict.find x st` specifies
  the location where the variable `x` is stored on the stack with respect to the 
  local stack frame.
- `class_name` is the name of the class that is to be generated.

The function returns a pair of the form `(cmds, size)`.
- `cmds` is a list of assembler commands,
- `size` is the maximum size of the stack that is needed. 

**Type safety:** In Python, Boolean expressions and arithmetical expressions are both nested tuples, so nothing prevents
us from calling `compile_bool` with an arithmetical expression.  In OCaml, they have the different types `bool_expr`
and `expr`, and the type checker rejects such a call.

In [ ]:
let rec compile_bool (expr : bool_expr) (st : (string, int) Dict.t) (class_name : string) : string list * int =
  match expr with
  | Compare (op, lhs, rhs) ->
      let op_to_cmd =
        match op with
        | Eq -> "if_icmpeq"
        | Ne -> "if_icmpne"
        | Le -> "if_icmple"
        | Ge -> "if_icmpge"
        | Lt -> "if_icmplt"
        | Gt -> "if_icmpgt"
      in
      let l1, sz1    = compile_expr lhs st class_name in
      let l2, sz2    = compile_expr rhs st class_name in
      let true_label = new_label () in
      let next_label = new_label () in
      let cmd_lst    = l1 @ l2 in
      let cmd        = op_to_cmd in
      let cmd_lst    = cmd_lst @ [ indent (cmd ^ " " ^ true_label) ] in
      let cmd_lst    = cmd_lst @ [ indent "bipush 0" ] in
      let cmd_lst    = cmd_lst @ [ indent ("goto " ^ next_label) ] in
      let cmd_lst    = cmd_lst @ [ String.make 4 ' ' ^ true_label ^ ":" ] in
      let cmd_lst    = cmd_lst @ [ indent "bipush 1" ] in
      let cmd_lst    = cmd_lst @ [ String.make 4 ' ' ^ next_label ^ ":" ] in
      (cmd_lst, max sz1 (1 + sz2))
  | And (lhs, rhs) | Or (lhs, rhs) ->
      let op_to_cmd = match expr with And _ -> "iand" | _ -> "ior" in
      let l1, sz1   = compile_bool lhs st class_name in
      let l2, sz2   = compile_bool rhs st class_name in
      let cmd       = op_to_cmd in
      let cmd_lst   = l1 @ l2 @ [ indent cmd ] in
      (cmd_lst, max sz1 (1 + sz2))
  | Not arg ->
      let l, sz   = compile_bool arg st class_name in
      let cmd_lst = [ indent "bipush 1" ] @ l @ [ indent "isub" ] in
      (cmd_lst, sz + 1)

Below is a test for the function `compile_bool`.

In [ ]:
let expr = Compare (Eq, Var "x", Var "y")
let st   = Dict.of_list [ ("x", 0); ("y", 1) ];;
compile_bool expr st "Sum"

In [ ]:
let expr = Not (Compare (Eq, Var "x", Var "y"))
let st   = Dict.of_list [ ("x", 0); ("y", 1) ];;
compile_bool expr st "Sum"

In [ ]:
let expr = Or (Compare (Eq, Var "x", Var "y"), Compare (Lt, Var "x", Var "y"))
let st   = Dict.of_list [ ("x", 0); ("y", 1) ];;
compile_bool expr st "Sum"

The function `compile_stmnt stmnt st class_name` takes three arguments:
- `stmnt` is an *abstract syntax tree* that represents a statement.  
  This abstract syntax tree has the type `stmnt`.  
- `st` is short for *symbol table*.  This is a dictionary that maps variable
  names to natural numbers.  Given a variable `x`, the number `Dict.find x st` specifies
  the location where the variable `x` is stored on the stack with respect to the 
  local stack frame.
- `class_name` is the name of the class that is to be generated.

The function returns a pair of the form `(cmds, size)`.
- `cmds` is a list of assembler commands,
- `size` is the maximum size of the stack that is needed. 

The Python version recognizes an expression statement by the fact that no other case matches.  In OCaml, an expression
statement is a value of the form `Expr_stmnt expr`.

In [ ]:
let rec compile_stmnt (stmnt : stmnt) (st : (string, int) Dict.t) (class_name : string) : string list * int =
  match stmnt with
  | Assign (var, expr) ->
      let cmd_lst, sz = compile_expr expr st class_name in
      let cmd_lst     = cmd_lst @ [ indent (Printf.sprintf "istore %d" (Dict.find var st)) ] in
      (cmd_lst, sz)
  | If (expr, sub_stmnt) ->
      let l1, sz1    = compile_bool expr st class_name in
      let l2, sz2    = compile_stmnt sub_stmnt st class_name in
      let else_label = new_label () in
      let lbl_stmnt  = String.make 4 ' ' ^ else_label ^ ":" in
      let cmd_lst    = l1 @ [ indent ("ifeq " ^ else_label) ] @ l2 @ [ lbl_stmnt ] in
      (cmd_lst, max sz1 sz2)
  | If_else (expr, then_stmnt, else_stmnt) ->
      let l1, sz1    = compile_bool expr st class_name in
      let l2, sz2    = compile_stmnt then_stmnt st class_name in
      let l3, sz3    = compile_stmnt else_stmnt st class_name in
      let else_label = new_label () in
      let next_label = new_label () in
      let if_stmnt   = indent ("ifeq " ^ else_label) in
      let else_stmnt = String.make 4 ' ' ^ else_label ^ ":" in
      let next_stmnt = String.make 4 ' ' ^ next_label ^ ":" in
      let goto_stmnt = indent ("goto " ^ next_label) in
      let cmd_lst    = l1 @ [ if_stmnt ] @ l2 @ [ goto_stmnt; else_stmnt ] @ l3 @ [ next_stmnt ] in
      (cmd_lst, max sz1 (max sz2 sz3))
  | While (expr, body_stmnt) ->
      let l1, sz1    = compile_bool expr st class_name in
      let l2, sz2    = compile_stmnt body_stmnt st class_name in
      let loop_label = new_label () in
      let next_label = new_label () in
      let if_stmnt   = indent ("ifeq " ^ next_label) in
      let loop_stmnt = String.make 4 ' ' ^ loop_label ^ ":" in
      let next_stmnt = String.make 4 ' ' ^ next_label ^ ":" in
      let goto_stmnt = indent ("goto " ^ loop_label) in
      let cmd_lst    = [ loop_stmnt ] @ l1 @ [ if_stmnt ] @ l2 @ [ goto_stmnt; next_stmnt ] in
      (cmd_lst, max sz1 sz2)
  | Return expr ->
      let cmd_lst, sz = compile_expr expr st class_name in
      let cmd_lst     = cmd_lst @ [ indent "ireturn" ] in
      (cmd_lst, sz)
  | Block stmnt_lst ->
      let cmd_lst = ref [] in
      let size    = ref 0 in
      List.iter
        (fun s ->
          let l, sz = compile_stmnt s st class_name in
          cmd_lst := !cmd_lst @ l;
          size    := max !size sz)
        stmnt_lst;
      (!cmd_lst, !size)
  | Expr_stmnt expr -> (* it is an expression statement *)
      compile_expr expr st class_name

In [ ]:
let stmnt = Assign ("x", Number "42")
let st    = Dict.of_list [ ("x", 0); ("y", 1) ];;
compile_stmnt stmnt st "Sum"

In [ ]:
let stmnt = If (Compare (Eq, Var "x", Var "y"), Assign ("x", Number "42"));;
compile_stmnt stmnt st "Sum"

In [ ]:
let stmnt = If_else (Compare (Lt, Var "x", Var "y"), Assign ("x", Var "y"), Assign ("y", Var "x"));;
compile_stmnt stmnt st "Sum"

In [ ]:
let stmnt = While (Compare (Lt, Var "x", Var "y"), Assign ("x", Arith (Add, Var "x", Number "1")));;
compile_stmnt stmnt st "Sum"

In [ ]:
let stmnt = Block [ Assign ("x", Var "y"); Assign ("x", Number "1"); Assign ("y", Var "x") ];;
compile_stmnt stmnt st "Sum"

In [ ]:
let compile_fct (fct_def : fct) (class_name : string) : string list * int =
  label_counter := 0;
  let { name; parameters; variables; stmnts } = fct_def in
  let m   = List.length parameters in
  let n   = List.length variables in
  let st  = ref Dict.empty in
  let cnt = ref 0 in
  List.iter
    (fun var ->
      st  := Dict.add var !cnt !st;
      cnt := !cnt + 1)
    (parameters @ variables);
  let cmd_lst = ref [] in
  let size    = ref 0 in
  List.iter
    (fun stmnt ->
      let l, sz = compile_stmnt stmnt !st class_name in
      cmd_lst := !cmd_lst @ l;
      size    := max !size sz)
    stmnts;
  let limit_locals = Printf.sprintf ".limit locals %d" (m + n) in
  let limit_stack  = Printf.sprintf ".limit stack  %d" !size in
  let return_stmnt = indent "return" in
  if name <> "main" then begin
    let meth    = Printf.sprintf ".method public static %s(%s)I" name (String.make m 'I') in
    let cmd_lst = [ meth; limit_locals; limit_stack ] @ !cmd_lst @ [ ".end method" ] in
    (cmd_lst, !size)
  end
  else begin
    let meth    = ".method public static main([Ljava/lang/String;)V" in
    let cmd_lst = [ meth; limit_locals; limit_stack ] @ !cmd_lst @ [ return_stmnt; ".end method" ] in
    (cmd_lst, !size)
  end

In [ ]:
let f = { name = "sum"; parameters = [ "x" ]; variables = [ "y"; "z" ]; stmnts = [ Return (Var "x") ] };;
compile_fct f "Sum"

In [ ]:
(* Instead of the Python module os.path, we use the module Filename of the OCaml standard library. *)

In [ ]:
let file = "~/Drive/Kurse/Formal-Languages/Python/Chapter-13/Examples/Primes.c";;
print_endline (Filename.dirname file);;
print_endline (Filename.basename file);;
print_endline (Filename.concat "abc" "xyz.c")

The input to `compile_program(file_name)` is the name of a `C` file.  This file assumed to have the ending `.c`.
This file is compiled and the resulting assembler file is written into a file with the same name but the ending `.jas`.

In [ ]:
let compile_program (file_name : string) : unit =
  let directory = Filename.dirname file_name in
  let base      = Filename.basename file_name in
  let base      = Filename.chop_suffix base ".c" in
  let outfile   = Filename.concat directory (base ^ ".jas") in
  let program   = read_file file_name in
  match parse_program program with
  | None -> ()   (* the syntax error has already been reported *)
  | Some fct_lst ->
      let cmd_lst = ref [] in
      List.iter
        (fun fct ->
          let l, _ = compile_fct fct base in
          cmd_lst := !cmd_lst @ l @ [ "\n" ])
        fct_lst;
      Out_channel.with_open_text outfile (fun handle ->
          output_string handle (".class public " ^ base ^ "\n");
          output_string handle ".super java/lang/Object\n\n";
          output_string handle ".method public <init>()V\n";
          output_string handle "    aload 0\n";
          output_string handle "    invokenonvirtual java/lang/Object/<init>()V\n";
          output_string handle "    return\n";
          output_string handle ".end method\n\n";
          List.iter (fun cmd -> output_string handle (cmd ^ "\n")) !cmd_lst)

The file `Primes.c` is a simple `C` program that computes the prime numbers in a naive way.

In [ ]:
Sys.chdir "Examples";;
Sys.getcwd ()

In [ ]:
shell "ls -l"

In [ ]:
cat "Primes.c"

In [ ]:
compile_program "Primes.c"

In [ ]:
shell "ls -l"

In [ ]:
cat "Primes.jas"

Next, we generate Java byte code using
[jasmin](http://jasmin.sourceforge.net).

In [ ]:
shell "jasmin Primes.jas"

Finally, we run the generated byte code.

In [ ]:
shell "ls -l"

In [ ]:
shell "java Primes"